In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l9.1 Capstone

Everything in one pipeline: **build** a modern PocketLM, **train** it, **evaluate**
it, **adapt** it with LoRA, **optimize** it with quantization and a KV cache, and
**serve** it, then write the engineering report that makes it all reproducible.
This is the model-engineer deliverable.

## 1. Build and train

A modern config (RMSNorm + SwiGLU + RoPE), trained on the corpus, measured against
the uniform-perplexity baseline.

In [ ]:
import copy
import json
from pocketlm import (CharTokenizer, PocketLM, PocketLMConfig, init_weights,
                      make_batch, estimate_loss, perplexity)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32, n_layer=2, n_head=4,
                     n_embd=64, norm="rmsnorm", mlp="swiglu", pos="rope")
model = init_weights(PocketLM(cfg))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(0)
for _ in range(150 if os.environ.get("POCKETLM_CI") else 500):
    x, y = make_batch(data, 32, 16, generator=g)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
ppl = perplexity(estimate_loss(model, data, 32, 16, iters=10, generator=g))
params = sum(p.numel() for p in model.parameters())
print(f"params {params}, perplexity {ppl:.2f} vs baseline {tok.vocab_size}")

## 2. Adapt with LoRA

Freeze the base, train a low-rank adapter, a fraction of the parameters.

In [ ]:
from pocketlm import add_lora, trainable_parameters

adapted = copy.deepcopy(model)
add_lora(adapted, rank=4)
trainable = sum(p.numel() for p in trainable_parameters(adapted))
aopt = torch.optim.AdamW(trainable_parameters(adapted), lr=1e-3)
for _ in range(40 if os.environ.get("POCKETLM_CI") else 100):
    x, y = make_batch(data, 32, 16, generator=g)
    _, loss = adapted(x, y)
    aopt.zero_grad(set_to_none=True)
    loss.backward()
    aopt.step()
print(f"LoRA trainable {trainable} / {params}  ({100 * trainable / params:.1f}%)")

## 3. Optimize: quantize + KV-cache parity

Measure 4-bit degradation, and confirm the cache that speeds serving is exact.

In [ ]:
from pocketlm import (fake_quantize, scaled_dot_product_attention, KVCache,
                      cached_step)

qmodel = copy.deepcopy(model)
with torch.no_grad():
    for p in qmodel.parameters():
        p.copy_(fake_quantize(p, bits=4))
q_ppl = perplexity(estimate_loss(qmodel, data, 32, 16, iters=10,
                                 generator=torch.Generator().manual_seed(0)))
qc, kc, vc = (torch.randn(1, 2, 5, 8) for _ in range(3))
full, _ = scaled_dot_product_attention(qc, kc, vc, causal=True)
cache = KVCache()
inc = torch.cat([cached_step(qc[:, :, t:t + 1], kc[:, :, t:t + 1], vc[:, :, t:t + 1], cache)
                 for t in range(5)], dim=2)
kv_parity = bool(torch.allclose(inc, full, atol=1e-5))
print(f"4-bit perplexity {q_ppl:.2f} (fp32 was {ppl:.2f}); KV-cache parity {kv_parity}")

## 4. Serve

Wrap the trained model behind the inference handler.

In [ ]:
from pocketlm import InferenceService

svc = InferenceService(model, tok)
resp = svc.handle({"prompt": "The ", "max_new_tokens": 30, "seed": 0})
print(resp["completion"])

## 5. Engineering report

The reproducible record: architecture, metrics vs baseline, adaptation, the
optimization trade, and honest limitations.

In [ ]:
report = {
    "artifact": "PocketLM (capstone)",
    "architecture": {"norm": "rmsnorm", "mlp": "swiglu", "pos": "rope",
                     "n_layer": cfg.n_layer, "n_head": cfg.n_head, "n_embd": cfg.n_embd},
    "params": params,
    "training": {"final_perplexity": round(ppl, 2), "uniform_baseline": tok.vocab_size},
    "adaptation": {"method": "LoRA", "trainable_params": trainable,
                   "trainable_pct": round(100 * trainable / params, 2)},
    "optimization": {"quantized_bits": 4, "quantized_perplexity": round(q_ppl, 2),
                     "kv_cache_parity": kv_parity},
    "serving": "InferenceService.handle({prompt, max_new_tokens})",
    "reproducibility": {"seed": 0, "corpus": "Gutenberg #100, 1MB slice"},
    "limitations": ["char-level tokenizer", "micro scale for CI", "single corpus"],
}
print(json.dumps(report, indent=2))

That is the full arc of the course, build, train, modernize, adapt, optimize, serve,
carried out on a model small enough to run in seconds yet faithful to how real
decoder-only LLMs are engineered.

In [ ]:
assert report["params"] > 0
assert ppl < tok.vocab_size                      # beat the uniform baseline
assert report["adaptation"]["trainable_pct"] < 20
assert report["optimization"]["kv_cache_parity"] is True
assert len(resp["completion"]) == len("The ") + 30
assert set(report) >= {"artifact", "architecture", "params", "training",
                       "adaptation", "optimization", "serving", "limitations"}